# שיעור 13 - זיכרון סוכן עם גרפי ידע של Cognee


## התקנה

מחברת זו מדגימה כיצד לבנות **עוזר קידוד** חכם עם זיכרון מתמשך באמצעות גרפי ידע של [**Cognee**](https://www.cognee.ai/) ומסגרת הסוכן של מיקרוסופט (MAF).

Cognee ממירה טקסט לא מובנה לגרף ידע מובנה ולשאילתא הנתמך בהטמעות וקטוריות — מה שנותן לסוכן שלך זיכרון לטווח ארוך עשיר ומודע למערכות יחסים.

### מה תלמדו
1. **בניית גרפי ידע**: המרת פרופילי מפתחים ושיטות עבודה מומלצות לידע מובנה וניתן לשאילתא.
2. **שילוב Cognee עם MAF**: שימוש בפונקציות `@tool` כדי לאפשר לסוכן MAF לשאול בגרף הידע של Cognee.
3. **שיחות עם מודעות מקרה**: שמירת ההקשר לאורך מספר שאלות באותה מפגש.
4. **זיכרון לטווח ארוך**: שמירת ידע חשוב לאורך מפגשים ושליפתו בשיחות חדשות.

### דרישות מוקדמות
- פייתון 3.9+
- Redis רץ מקומית (`docker run -d -p 6379:6379 redis`) לניהול מפגשים
- מפתח API ל-LLM (למשל OpenAI) — הגדר `LLM_API_KEY` בקובץ `.env`
- `CACHING=true` בקובץ `.env` (נדרש למפגשי Cognee)
- פרויקט Azure AI Foundry עם מודל צ'אט פרוס
- `AZURE_AI_PROJECT_ENDPOINT` ו-`AZURE_AI_MODEL_DEPLOYMENT_NAME` בקובץ `.env`
- Azure CLI מאומת (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## סוגי זיכרון סוכן

המחברת הזו חוקרת את אותם שלושה סוגי זיכרון מהמחברת הראשית של שיעור 13, אך משתמשת ב-Cognee כזיכרון לטווח הארוך:

| סוג זיכרון | מנגנון | אורך חיים |
|---|---|---|
| **עובד** | `agent.create_session()` (MAF) | שיחה יחידה |
| **לטווח קצר** | מטמון סשן של Cognee (Redis) | סשן יחיד |
| **לטווח ארוך** | גרף הידע של Cognee + וקטורים | קבוע |

### ארכיטקטורת הזיכרון של Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## הכנת אחסון קוגני


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## חלק 1 — בניית בסיס הידע

אנו מקבלים שלושה סוגי נתונים כדי ליצור בסיס ידע מקיף עבור העוזר התכנותי שלנו:

1. **פרופיל המפתח** — מומחיות אישית ורקע טכני  
2. **שיטות עבודה מומלצות בפייתון** — הזן של פייתון עם הנחיות מעשיות  
3. **שיחות היסטוריות** — סשני שאלות ותשובות קודמים בין מפתחים ועוזרי בינה מלאכותית


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## ויזואליזציה של גרף הידע

Cognee יכול להציג ויזואליזציה אינטראקטיבית ב-HTML של הישויות והקשרים שחלץ.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## להעשיר זיכרון עם Memify

`memify()` מנתחת את גרף הידע ומייצרת כללים אינטיליגנטיים — זיהוי תבניות, שיטות מומלצות, ויחסים בין מושגים.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## חלק 2 — סוכן MAF עם כלי Cognee

כעת אנו יוצרים סוכן MAF שיכול לשאול את גרף הידע של Cognee דרך פונקציות `@tool`. זה מאפשר לסוכן לנצל את מלוא הכוח של חיפוש סמנטי המודע לגרף תוך שהוא שומר על הקשר שיחתי באמצעות מפגשים.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## זיכרון עבודה עם מושבים

ה-`AgentSession` (נוצר דרך `agent.create_session()`) מספק זיכרון עבודה בתוך מושב. הסוכן יכול להתייחס להודעות קודמות וגם לשאול את גרף הידע לטווח ארוך של Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## מפגש חדש — זיכרון לטווח ארוך נשמר

התחלת מפגש חדש מנקה את הזיכרון הפעיל, אך גרף הידע של Cognee עדיין זמין. הסוכן יכול לשלוף את אותו הידע לטווח הארוך בשיחה חדשה לחלוטין.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## סיכום

במחברת זו בנית עוזר קידוד המשלב את **זיכרון העבודה של MAF** (`agent.create_session()`) עם **גרף הידע עתיר-הזמן של Cognee**.

### מה שלמדת
1. **בניית גרף ידע**: Cognee סופג טקסט לא מובנה ובונה גרף + זיכרון וקטורי.
2. **העשרת הגרף עם memify**: עובדות נגזרות ויחסים עשירים יותר מעל הגרף הקיים שלך.
3. **אינטגרציה בין MAF ל-Cognee**: פונקציות `@tool` מאפשרות לסוכן MAF לשאול את גרף Cognee בצורה טבעית.
4. **זיכרון עבודה + זיכרון ארוך טווח**: `AgentSession` (דרך `agent.create_session()`) מספק הקשר מושב בעוד ש-Cognee מספק ידע מתמשך.
5. **חיפוש מסונן עם NodeSets**: ייעוד תת-קבוצות ספציפיות של גרף הידע (למשל רק עקרונות).

### עיקרי הדברים
- **Cognee** ממיר טקסט גולמי לזיכרון מובנה, מודע ליחסים — חזק יותר מאחסון וקטורי שטוח.
- פונקציות **`@tool`** מהוות גשר נקי בין סוכני MAF למערכות ידע חיצוניות.
- **`AgentSession`** (דרך `agent.create_session()`) שומר הקשר לכל שיחה בנפרד מהידע האצור לאורך זמן.
- אותו גרף ידע משרת מספר מושבים וסוכנים.

### יישומים מעשיים
- **שותפים לפיתוח לתמיכה בקוד**: סקירת קוד, ניתוח תקלות, עוזרי ארכיטקטורה
- **שותפים למול לקוחות**: סוכני תמיכה המתבססים על מסמכי מוצר, שאלות נפוצות, והערות CRM
- **שותפים מומחים פנימיים**: עוזרי מדיניות, משפט וביטחון המבצעים ניתוח על פי הנחיות
- **שכבות נתונים מאוחדות**: שילוב נתונים מובנים ובלתי מובנים לגרף שאפשר לשאול אותו

### השלבים הבאים
- להתנסות במודעות זמנית ב-Cognee
- להגדיר אונטולוגיית OWL לאיכות גרף תחומית
- להוסיף לולאות משוב משתמש לשיפור אחזור לאורך זמן
- להרחיב למערכות מולטי-סוכן שמשתפות את אותה שכבת זיכרון Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**הבהרה**:  
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון כי תרגומים אוטומטיים עלולים להכיל טעויות או אי-דיוקים. המסמך המקורי בשפת המקור צריך להיחשב כמקור הסמכות. למידע קריטי מומלץ לתרגם בשירות תרגום מקצועי אדם. איננו אחראים לכל אי-הבנות או פרשנויות מוטעות הנובעות מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
